# Billing — extracción paso a paso (versión notebook)

Copia **visual y ejecutable celda por celda** de `extract_run_billing.py`.
El objetivo es poder **mostrar y explicar el proceso completo** para un run:

> "¿Cuánto costó este run y de dónde salió cada número?"

**Fuentes (todas verificables):**

| Dato | Fuente |
| --- | --- |
| run, jobs, timestamps | GitHub Actions REST API |
| duración total / billable | Actions **timing** API |
| action_minutes | `gh aw audit` / `gh aw logs` |
| AIC (AI Credits) | artefacto `usage` de gh-aw + `models.json` |
| precios Actions | docs oficiales de GitHub |
| `1 AIC = 0.01 USD` | gh-aw AI Credits Specification sección 3.1 |

> Principios: **no redondear**, distinguir `null` (sin dato) de `0`, y guardar la evidencia cruda.


## Paso 0 — Configuración

Todo lo que es *supuesto o fuente externa* queda explícito y separado del dato observado.

In [ ]:
import csv
import json
import math
import os
import subprocess
import time
from datetime import datetime, timezone
from pathlib import Path
from IPython.display import display

# --- Qué vamos a analizar ---------------------------------------------------
REPOSITORY = "vaadin/flow"
RUN_IDS = ["36728232182"]        # cambia por el run que quieras inspeccionar
OUT_ROOT = Path("billing/output")  # misma ruta que usa el script .py
SKIP_GHAW = False                  # True = no llamar a `gh aw logs` (solo Actions)

# --- Precios oficiales de GitHub Actions (USD / minuto) ---------------------
# Fuente: docs.github.com -> billing -> about-billing-for-github-actions
ACTIONS_RATES_USD_PER_MIN = {
    "actions_linux_slim": 0.002,
    "actions_linux": 0.006,
    "actions_linux_arm": 0.005,
    "actions_windows": 0.010,
    "actions_windows_arm": 0.010,
    "actions_macos": 0.062,
}

# --- Mapeo EXPLÍCITO label de runner -> SKU de facturación ------------------
# Si no hay match => sku = None (NO inventar tarifa).
RUNNER_LABEL_TO_SKU = {
    "ubuntu-slim": "actions_linux_slim",
    "ubuntu-latest": "actions_linux",
    "ubuntu-24.04": "actions_linux",
    "ubuntu-22.04": "actions_linux",
    "ubuntu-24.04-arm": "actions_linux_arm",
    "ubuntu-22.04-arm": "actions_linux_arm",
    "windows-latest": "actions_windows",
    "windows-2022": "actions_windows",
    "windows-2025": "actions_windows",
    "windows-11-arm": "actions_windows_arm",
    "macos-latest": "actions_macos",
    "macos-14": "actions_macos",
    "macos-15": "actions_macos",
    "macos-13": "actions_macos",
}

# --- Definición normativa de AI Credits (gh-aw) -----------------------------
AIC_USD = 0.01  # AI Credits Specification sección 3.1

print("Repo :", REPOSITORY)
print("Runs :", RUN_IDS)
print("Salida:", OUT_ROOT)


## Paso 1 — Utilidades

Envoltorios para ejecutar comandos y guardar JSON crudo (trazabilidad).

In [ ]:
def run_cmd(args, stdin_text=None, timeout=600):
    """Ejecuta un comando limpiando variables que ensucian la salida."""
    env = os.environ.copy()
    env.update({"NO_COLOR": "1", "GH_PAGER": "cat", "PAGER": "cat", "TERM": "dumb"})
    started = time.perf_counter()
    proc = subprocess.run(args, input=stdin_text, capture_output=True,
                          text=True, timeout=timeout, env=env)
    return {
        "args": list(args),
        "returncode": proc.returncode,
        "stdout": proc.stdout,
        "stderr": proc.stderr,
        "elapsed_seconds": time.perf_counter() - started,
    }


def gh_api_json(endpoint, out_path=None):
    """GET a la API vía `gh api`. Guarda la respuesta cruda si se indica ruta."""
    res = run_cmd(["gh", "api", endpoint], timeout=180)
    if res["returncode"] != 0:
        raise RuntimeError(f"gh api {endpoint} fallo: {res['stderr']}")
    data = json.loads(res["stdout"])
    if out_path is not None:
        write_json(out_path, data)
    return data


def write_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, ensure_ascii=False), encoding="utf-8")


def iso_to_dt(value):
    if not value:
        return None
    return datetime.fromisoformat(value.replace("Z", "+00:00"))


print("Utilidades listas.")


## Paso 2 — GitHub Actions (fuente de verdad del compute)

Traemos y guardamos: `run.json`, `jobs.json`, `timing.json`, `artifacts.json`.

In [ ]:
RUN_ID = RUN_IDS[0]
RUN_DIR = OUT_ROOT / REPOSITORY.replace("/", "__") / f"run-{RUN_ID}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

run = gh_api_json(f"repos/{REPOSITORY}/actions/runs/{RUN_ID}", RUN_DIR / "run.json")
jobs_payload = gh_api_json(f"repos/{REPOSITORY}/actions/runs/{RUN_ID}/jobs?per_page=100",
                           RUN_DIR / "jobs.json")
jobs = jobs_payload.get("jobs", [])

timing = None
try:
    timing = gh_api_json(f"repos/{REPOSITORY}/actions/runs/{RUN_ID}/timing",
                         RUN_DIR / "timing.json")
except RuntimeError as exc:
    print("WARN timing no disponible:", exc)

try:
    gh_api_json(f"repos/{REPOSITORY}/actions/runs/{RUN_ID}/artifacts",
                RUN_DIR / "artifacts.json")
except RuntimeError:
    pass

print("run_duration_ms (timing API):", (timing or {}).get("run_duration_ms"))
print("billable por SO            :", (timing or {}).get("billable"))
print("repo privado?               :", bool((run.get("repository") or {}).get("private")))
print()
print("Jobs (%d):" % len(jobs))
for j in jobs:
    print(" -", j["name"], "|", j["conclusion"], "|", j.get("labels"),
          "|", j.get("started_at"), "->", j.get("completed_at"))


## Paso 3 — Análisis de jobs y costo de Actions

Por cada job: duración observada, runner **mapeado explícitamente** a un SKU, y costo nocional.

> `actions_cost_usd_notional` = precio de lista. Para repo **público**, lo **facturado** es `0`.

In [ ]:
def infer_sku_from_job(job):
    labels = [str(x).lower() for x in (job.get("labels") or [])]
    for label in labels:
        if label in RUNNER_LABEL_TO_SKU:
            return RUNNER_LABEL_TO_SKU[label], label
    if any("self-hosted" in x for x in labels):
        return None, (labels[0] if labels else None)
    return None, (labels[0] if labels else None)


def analyze_jobs(jobs):
    analyzed, total_seconds, total_ceil, total_notional, skus = [], 0.0, 0.0, 0.0, set()
    for job in jobs:
        if job.get("status") != "completed":
            continue
        started, completed = iso_to_dt(job.get("started_at")), iso_to_dt(job.get("completed_at"))
        seconds = (completed - started).total_seconds() if (started and completed and completed >= started) else None
        if job.get("conclusion") == "skipped" and not seconds:
            seconds = 0.0

        sku, matched = infer_sku_from_job(job)
        rate = ACTIONS_RATES_USD_PER_MIN.get(sku) if sku else None
        ceil_minutes = notional = None
        if seconds is not None:
            total_seconds += seconds
            ceil_minutes = math.ceil(seconds / 60.0) if seconds > 0 else 0.0
            total_ceil += ceil_minutes
            if rate is not None:
                notional = ceil_minutes * rate
                total_notional += notional
        if sku:
            skus.add(sku)

        analyzed.append({
            "job_name": job.get("name"), "conclusion": job.get("conclusion"),
            "started_at": job.get("started_at"), "completed_at": job.get("completed_at"),
            "duration_seconds": seconds, "runner_name": job.get("runner_name"),
            "runner_labels_raw": json.dumps(job.get("labels") or []),
            "runner_label_matched": matched, "runner_sku": sku,
            "rate_usd_per_min": rate, "ceil_minutes": ceil_minutes,
            "actions_cost_usd_notional": notional,
        })
    return analyzed, {"actions_vm_seconds": total_seconds,
                      "actions_ceil_minutes": total_ceil,
                      "actions_cost_usd_notional": total_notional,
                      "runner_skus": sorted(skus)}


job_rows, agg = analyze_jobs(jobs)
for r in job_rows:
    print(f"{r['job_name']:<12} {r['conclusion']:<9} {r['duration_seconds']!s:>6}s "
          f"{str(r['runner_label_matched']):<14} -> {str(r['runner_sku']):<20} "
          f"${r['actions_cost_usd_notional']}")
print()
print("Agregado:", agg)


## Paso 4 — Inferencia de IA (AIC / tokens)

Se descarga el artefacto `usage` con `gh aw logs`. Si no hay dato, se deja **`None`** (sin dato), nunca `0`.

In [ ]:
def fetch_ai_usage(repo, run_id, workdir):
    """
    Intenta obtener AIC/tokens. Devuelve dict con provenance.

    Estrategia: `gh aw logs --stdin --artifacts usage` y busca el artefacto
    de uso descargado. NO inventa: si no aparece, devuelve nulls.
    """
    out_dir = Path(workdir) / "gh-aw"
    out_dir.mkdir(parents=True, exist_ok=True)

    res = run_cmd(
        ["gh", "aw", "logs", "--stdin", "--repo", repo, "--json",
         "--output", str(out_dir), "--artifacts", "usage"],
        stdin_text=f"{run_id}\n",
        timeout=1800,
    )
    (out_dir / "gh-aw-logs.stdout.txt").write_text(res["stdout"] or "", encoding="utf-8")
    (out_dir / "gh-aw-logs.stderr.txt").write_text(res["stderr"] or "", encoding="utf-8")
    write_json(out_dir / "gh-aw-logs.command.json", {
        "args": res["args"], "returncode": res["returncode"],
        "elapsed_seconds": res["elapsed_seconds"],
    })

    result = {
        "aic": None,
        "input_tokens": None,
        "output_tokens": None,
        "cache_read_tokens": None,
        "cache_write_tokens": None,
        "reasoning_tokens": None,
        "ai_request_count": None,
        "ai_models": None,
        "aic_from_records": None,
        "ai_provenance": None,
    }

    # ------------------------------------------------------------------
    # FUENTE PREFERIDA (autoritativa de gh-aw):
    #   run_summary.json -> token_usage_summary (trae todos los tokens + total_aic)
    # ------------------------------------------------------------------
    for rsum in out_dir.rglob("run_summary.json"):
        try:
            data = json.loads(rsum.read_text(encoding="utf-8", errors="replace"))
        except json.JSONDecodeError:
            continue
        if str(data.get("run_id")) != str(run_id):
            continue
        tus = data.get("token_usage_summary")
        if not isinstance(tus, dict):
            continue
        result["aic"] = tus.get("total_aic")
        result["input_tokens"] = tus.get("total_input_tokens")
        result["output_tokens"] = tus.get("total_output_tokens")
        result["cache_read_tokens"] = tus.get("total_cache_read_tokens")
        result["cache_write_tokens"] = tus.get("total_cache_write_tokens")
        result["reasoning_tokens"] = tus.get("total_reasoning_tokens")  # puede no existir
        result["ai_request_count"] = tus.get("total_requests")
        by_model = tus.get("by_model") or {}
        result["ai_models"] = ",".join(sorted(by_model.keys())) or None
        result["ai_provenance"] = (
            "gh aw run_summary.json .token_usage_summary "
            f"(modelos: {result['ai_models']})"
        )
        break

    # ------------------------------------------------------------------
    # FALLBACK 1: registro del JSON de `gh aw logs` (aic reportado)
    # ------------------------------------------------------------------
    payload = None
    try:
        payload = json.loads(res["stdout"])
    except (json.JSONDecodeError, TypeError):
        payload = None

    record = None
    if isinstance(payload, dict):
        for key in ("per_run_breakdown", "runs"):
            items = payload.get(key)
            if isinstance(items, list):
                for item in items:
                    if str(item.get("run_id")) == str(run_id):
                        record = item
                        break
            if record:
                break

    if result["aic"] is None and record:
        for candidate in ("aic", "total_aic", "ai_credits", "ai_credits_total"):
            if record.get(candidate) is not None:
                result["aic"] = float(record[candidate])
                result["ai_provenance"] = f"gh aw logs json .{candidate}"
                break
    if record:
        for cand in ("input_tokens", "output_tokens", "cache_read_tokens",
                     "cache_write_tokens", "reasoning_tokens"):
            if result[cand] is None and record.get(cand) is not None:
                result[cand] = record[cand]

    # ------------------------------------------------------------------
    # FALLBACK 2: agregar token_usage.jsonl (agent/detection/evals)
    #   OJO: el nombre real del archivo usa guion bajo.
    #   Se ejecuta SIEMPRE para dejar aic_from_records (reconciliación),
    #   pero solo rellena aic/tokens si la fuente preferida no los trajo.
    # ------------------------------------------------------------------
    agg = {"input_tokens": 0, "output_tokens": 0,
           "cache_read_tokens": 0, "cache_write_tokens": 0,
           "reasoning_tokens": 0}
    aic_sum = 0.0
    requests = 0
    models = set()
    found = False
    for jsonl in out_dir.rglob("token_usage.jsonl"):
        if jsonl.parent.name != "agent":  # mismo alcance que run_summary (agente)
            continue
        for line in jsonl.read_text(encoding="utf-8", errors="replace").splitlines():
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            found = True
            requests += 1
            for k in agg:
                agg[k] += rec.get(k) or 0
            aic_sum += rec.get("ai_credits_this_response") or 0
            if rec.get("model"):
                models.add(rec["model"])
    if found:
        if result["input_tokens"] is None:
            result.update(agg)
        result["aic_from_records"] = aic_sum
        if result["ai_request_count"] is None:
            result["ai_request_count"] = requests
        if result["ai_models"] is None:
            result["ai_models"] = ",".join(sorted(models)) or None
        if result["aic"] is None:
            result["aic"] = aic_sum
        if result["ai_provenance"] is None:
            result["ai_provenance"] = "token_usage.jsonl (agregado)"

    # Los tokens de razonamiento no suelen reportarse: spec sección 3.7 dice 0.
    if result["reasoning_tokens"] is None and result["input_tokens"] is not None:
        result["reasoning_tokens"] = 0

    result["aic_usd"] = None if result["aic"] is None else float(result["aic"]) * AIC_USD
    return result

ai = ({"aic": None, "aic_usd": None, "input_tokens": None, "output_tokens": None,
       "cache_read_tokens": None, "cache_write_tokens": None,
       "reasoning_tokens": None, "ai_provenance": None}
      if SKIP_GHAW else fetch_ai_usage(REPOSITORY, RUN_ID, RUN_DIR))

print("AIC        :", ai["aic"])
print("AIC -> USD :", ai["aic_usd"], "(= aic × 0.01 según spec sección 3.1)")
print("Procedencia:", ai["ai_provenance"])


## Paso 5 — Fila consolidada del run

Cada columna es **observada** (API) o **calculada** (marcada por nombre y procedencia).

In [ ]:
is_private = bool((run.get("repository") or {}).get("private"))
billable = None
if timing and timing.get("billable"):
    billable = {k: {"total_ms": v.get("total_ms"), "jobs": v.get("jobs")}
                for k, v in timing["billable"].items()}

row = {
    "repo": REPOSITORY, "run_id": run.get("id"), "run_attempt": run.get("run_attempt"),
    "workflow_id": run.get("workflow_id"), "workflow_name": run.get("name"),
    "event": run.get("event"), "status": run.get("status"), "conclusion": run.get("conclusion"),
    "created_at": run.get("created_at"), "run_started_at": run.get("run_started_at"),
    "updated_at": run.get("updated_at"), "head_branch": run.get("head_branch"),
    "head_sha": run.get("head_sha"), "html_url": run.get("html_url"),
    "repo_private": is_private,
    # Actions
    "run_duration_ms_github": (timing or {}).get("run_duration_ms"),
    "actions_billable_by_os_json": json.dumps(billable) if billable else None,
    "jobs_count": len(jobs),
    "actions_vm_seconds": agg["actions_vm_seconds"],
    "actions_ceil_minutes": agg["actions_ceil_minutes"],
    "actions_cost_usd_notional": agg["actions_cost_usd_notional"],
    "actions_cost_usd_actual": agg["actions_cost_usd_notional"] if is_private else 0.0,
    "runner_skus_json": json.dumps(agg["runner_skus"]),
    # IA
    "aic": ai["aic"], "aic_usd": ai["aic_usd"],
    "input_tokens": ai["input_tokens"], "output_tokens": ai["output_tokens"],
    "cache_read_tokens": ai["cache_read_tokens"], "cache_write_tokens": ai["cache_write_tokens"],
    "reasoning_tokens": ai["reasoning_tokens"], "ai_provenance": ai["ai_provenance"],
    # Combinado (solo si hay ambos)
    "estimated_total_cost_usd": (None if ai["aic_usd"] is None
                                 else agg["actions_cost_usd_notional"] + ai["aic_usd"]),
}

for k, v in row.items():
    print(f"{k}: {v}")


## Paso 6 — Guardar CSV y evidencia

Sin redondeo: los valores se escriben tal como se obtuvieron/calcularon.

In [ ]:
out_dir = OUT_ROOT / REPOSITORY.replace("/", "__")
out_dir.mkdir(parents=True, exist_ok=True)

csv_path = out_dir / "billing_runs.csv"
with csv_path.open("w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=list(row.keys()))
    writer.writeheader()
    writer.writerow(row)

jobs_csv = out_dir / "billing_jobs.csv"
with jobs_csv.open("w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=list(job_rows[0].keys()))
    writer.writeheader()
    writer.writerows(job_rows)

write_json(RUN_DIR / "billing_row.json", row)
write_json(RUN_DIR / "job_analysis.json", job_rows)

print("Escrito:", csv_path)
print("Escrito:", jobs_csv)
print("Evidencia cruda en:", RUN_DIR)


## Paso 7 — Tabla de procedencia (resumen para la reunión)

| Columna | Tipo | Fuente | Transformación |
| --- | --- | --- | --- |
| `run_id`, `workflow_id`, `event`, `status`, `conclusion`, timestamps | observado | Actions API `/runs/{id}` | ninguna |
| `run_duration_ms_github` | observado | Actions **timing** API | ninguna |
| `actions_billable_by_os_json` | observado | Actions **timing** API | ninguna (⚠️ 0 en repos sin acceso/públicos) |
| `duration_seconds` (por job) | calculado | `/jobs` (`completed_at - started_at`) | resta de timestamps |
| `runner_sku` | calculado | label de runner | mapeo **explícito** (Paso 0) |
| `actions_cost_usd_notional` | calculado | `ceil(minutos) × tarifa` | precio de lista |
| `actions_cost_usd_actual` | derivado | repo público ⇒ 0 | doc oficial GitHub |
| `aic` | observado | artefacto `usage` gh-aw | ninguna |
| `aic_usd` | calculado | `aic × 0.01` | spec sección 3.1 |
| `estimated_total_cost_usd` | calculado | Actions nocional + AIC | solo si ambos existen |

### Advertencias
- `aic = None` = **sin dato** (no "cero").
- Repo **público** ⇒ `actions_cost_usd_actual = 0` (Actions gratis para runners estándar).
- Ningún valor se redondea antes de guardarse.
